# Partitions - All Points (Sea States)

Partition reconstructed spectra at **all sites/coordinates** using PTM1. Input: 100 sea states (clusters), no dates.

**Output (per grid, all sites):**
- **Full spectrum** (from full spectrum before partitioning): hs, tp, tm02, dp, dm
- **Partitioned** (PTM1 swells=1 → 2 parts): phs0, phs1, ptp0, ptp1, pdp0, pdp1, spr0, spr1

**Prerequisites:**
1. **Create input spectra** (if grid{N}/inputs/spectra_point_{N}.nc are missing or all zeros):  
   `python create_cyclone_input_spectra.py` — samples 49 sea states from ds_predicted and builds JONSWAP spectra.
2. Run `reconstruct_spectra.py --grid N --local` for each grid.

**Wind for PTM1:** Uses wind (Windv_x, Windv_y) from ds_predicted_test_all.nc per point (grid1→point_1, grid2→point_2, etc.).

**Spr:** Computed from PTM1 partitioning (spec.dspr()), same as 00_Partitions_cluster_points.

In [1]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import warnings
import wavespectra  # registers .spec accessor

In [2]:
data = xr.open_dataset('/lustre/geocean/WORK/users/montanoj/personal/Cyclones_NC/outputs/partitions_cyclones/dm_grid1_cyclone_0.nc')

In [3]:
data

<xarray.Dataset> Size: 440kB
Dimensions:  (time: 24, site: 1895)
Coordinates:
  * time     (time) int64 192B 0 1 2 3 4 5 6 7 8 ... 15 16 17 18 19 20 21 22 23
  * site     (site) int64 15kB 1 2 3 4 5 6 7 ... 1890 1891 1892 1893 1894 1895
    lat      (site) float64 15kB ...
    lon      (site) float64 15kB ...
    coord_x  (site) float64 15kB ...
    coord_y  (site) float64 15kB ...
Data variables:
    dm       (time, site) float64 364kB ...

In [ ]:
# Create input spectra from cyclone data (run if diagnostic showed "Spectrum has no energy")
# Populates grid{N}/inputs/spectra_point_{N}.nc with 49 JONSWAP spectra from ds_predicted
import subprocess
import sys
need_create = False
for g in ["grid1", "grid2", "grid3", "grid4"]:
    p = os.path.join(os.path.abspath("."), g, "inputs", f"spectra_point_{g[-1]}.nc")
    if not os.path.exists(p):
        need_create = True
        break
    try:
        with xr.open_dataset(p) as ds:
            v = list(ds.data_vars)[0]
            if ds[v].values.max() < 1e-10:
                need_create = True
                break
    except Exception:
        need_create = True
        break
if need_create:
    print("Creating input spectra from cyclone data...")
    result = subprocess.run([sys.executable, "create_cyclone_input_spectra.py"], capture_output=True, text=True, cwd=os.path.abspath("."))
    print(result.stdout if result.returncode == 0 else (result.stderr or result.stdout)[:800])
    if result.returncode != 0:
        print("Run manually: python create_cyclone_input_spectra.py")
else:
    print("Input spectra exist with non-zero energy. Skip creation.")

In [ ]:


warnings.filterwarnings("ignore", category=RuntimeWarning)

REGION_NAME = "NorthCarolina"
BASE_DIR = os.path.abspath(".")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs", "partitions_cyclones")
os.makedirs(OUTPUT_DIR, exist_ok=True)

GEBCO_FILE = os.path.join(BASE_DIR, "inputs", "gebco_bathymetry.nc")
WIND_DATA_PATH = os.path.join(BASE_DIR, "inputs", "predicted_syntetic_all_OK_5vars_cyclone_id.nc")

TARGET_POINTS = {
    "grid1": {"label": "grid1", "lat": 33.441, "lon": -77.766},
    "grid2": {"label": "grid2", "lat": 33.8928, "lon": -76.9845},
    "grid3": {"label": "grid3", "lat": 35.10, "lon": -75.36},
    "grid4": {"label": "grid4", "lat": 36.603, "lon": -74.837},
}

# Map each grid to its point in ds_predicted (wind comes from that point)
GRID_TO_POINT = {"grid1": "point_1", "grid2": "point_2", "grid3": "point_3", "grid4": "point_4"}

# Full spectrum (non-partitioned): hs, tp, tm02, dp, dm
FULL_SPECTRUM_OUTPUTS = [
    ("hs", "hs", "significant_wave_height", "m"),
    ("tp", "tp", "peak_period", "s"),
    ("tm02", "tm02", "mean_period", "s"),
    ("dp", "dpm", "peak_direction", "degrees"),
    ("dm", "dm", "mean_direction", "degrees"),
]
# Partitioned (2 parts): phs0/1, ptp0/1, pdp0/1, spr0/1
PART_OUTPUTS = ["phs", "ptp", "pdp", "spr"]

print(f"Output dir: {OUTPUT_DIR}")
print(f"Loading wind data from {WIND_DATA_PATH}")
wind_ds = xr.open_dataset(WIND_DATA_PATH)
n_cases = wind_ds.sizes.get("cyclone_id", wind_ds.sizes.get("time", 0))
print(f"  Wind data: cyclone_id x time x point, {list(wind_ds.data_vars)}")

print(f"\nLoading GEBCO bathymetry...")
gebco = xr.open_dataset(GEBCO_FILE)

In [ ]:
def get_depth_from_gebco(lon_deg: float, lat_deg: float) -> float:
    """Return (positive) water depth [m] from GEBCO nearest to (lon, lat)."""
    depth_var = None
    for var in gebco.data_vars:
        vl = var.lower()
        if "elevation" in vl or "depth" in vl or "bathymetry" in vl or vl == "z":
            depth_var = var
            break
    if depth_var is None:
        depth_var = list(gebco.data_vars.keys())[0]

    gebco_lon = next((c for c in gebco.coords if "lon" in c.lower() or c == "x"), None)
    gebco_lat = next((c for c in gebco.coords if "lat" in c.lower() or c == "y"), None)
    if gebco_lon is None or gebco_lat is None:
        raise ValueError("Could not find lon/lat in GEBCO")

    depth = gebco[depth_var]
    if depth.min() < 0:
        depth = np.abs(depth)
    depth_at_point = depth.sel(
        {gebco_lon: lon_deg, gebco_lat: lat_deg},
        method="nearest",
    )
    return float(depth_at_point.values)

In [ ]:
def get_wind_from_cyclone_data(wind_ds, grid_name, n_cyclones):
    """Get wspd and wdir for each sea state from ds_predicted_test_all.nc.

    Wind is taken from the point matching the grid (grid1->point_1, etc.).
    Spectra time order is case-major, time-minor; wind is flattened the same way.
    Returns DataArrays with dim time and values (n_cyclones,).
    """
    point_name = GRID_TO_POINT[grid_name]
    if "Windv_x" not in wind_ds.data_vars or "Windv_y" not in wind_ds.data_vars:
        raise ValueError("Wind data must have Windv_x and Windv_y")

    # Select wind for this point; dims typically (case_num, time, point)
    point_dim = "point" if "point" in wind_ds.dims else next((d for d in wind_ds.dims if "point" in d.lower()), None)
    if point_dim is None:
        raise ValueError("Wind data must have a 'point' dimension")
    # Handle string coords (point_1, ...) or integer index (0->point_1, 1->point_2)
    pt_idx = point_name.split("_")[-1] if "_" in point_name else point_name
    try:
        pt_idx = int(pt_idx) - 1  # point_1 -> 0
    except ValueError:
        pass
    try:
        vx = wind_ds["Windv_x"].sel(**{point_dim: point_name}, method="nearest").values
        vy = wind_ds["Windv_y"].sel(**{point_dim: point_name}, method="nearest").values
    except (KeyError, TypeError):
        vx = wind_ds["Windv_x"].isel(**{point_dim: pt_idx}).values
        vy = wind_ds["Windv_y"].isel(**{point_dim: pt_idx}).values

    # Flatten to (n,) in case-major, time-minor order
    vx_flat = np.asarray(vx).flatten()
    vy_flat = np.asarray(vy).flatten()

    # Limit to n_cyclones (spectra may have fewer)
    n = min(len(vx_flat), n_cyclones)
    vx_flat = vx_flat[:n]
    vy_flat = vy_flat[:n]

    wspd = np.sqrt(vx_flat**2 + vy_flat**2)
    # Meteorological convention: direction FROM which wind blows (0=N, 90=E)
    wdir = (270 - np.degrees(np.arctan2(vy_flat, vx_flat))) % 360.0

    wspd_da = xr.DataArray(wspd, dims=["time"], coords={"time": np.arange(n)})
    wdir_da = xr.DataArray(wdir, dims=["time"], coords={"time": np.arange(n)})
    return wspd_da, wdir_da

In [ ]:
def process_grid_cyclones(grid_name):
    """Partition reconstructed spectra for one grid. Input: sea states (no dates).
    Outputs: full spectrum (hs, tp, tm02, dp, dm) and partitioned (phs0/1, ptp0/1, pdp0/1, spr0/1)."""
    spectra_path = os.path.join(BASE_DIR, f"{grid_name}", "outputs", f"reconstructed_spectra_{grid_name}_cyclones.nc")
    if not os.path.exists(spectra_path):
        raise FileNotFoundError(f"Run reconstruct_spectra.py --grid {grid_name[-1]} --local first. Missing: {spectra_path}")

    print(f"\n{'='*60}")
    print(f"Processing {grid_name}")
    print(f"{'='*60}")

    ds = xr.open_dataset(spectra_path)
    efth_var = "efth" if "efth" in ds.data_vars else list(ds.data_vars)[0]
    spectra = ds.rename({efth_var: "efth"})

    n_cyclones = spectra.sizes["time"]
    n_sites = spectra.sizes["site"]
    print(f"  Sea states: {n_cyclones}, Sites: {n_sites}")

    wspd, wdir = get_wind_from_cyclone_data(wind_ds, grid_name, n_cyclones)

    # Get depth per site from GEBCO
    if "coord_x" in spectra.coords and "coord_y" in spectra.coords:
        lons = spectra.coord_x.values
        lats = spectra.coord_y.values
    elif "lon" in spectra.coords and "lat" in spectra.coords:
        lons, lats = spectra.lon.values, spectra.lat.values
    else:
        raise ValueError("Spectra must have coord_x/coord_y or lon/lat")

    depth_values = np.array([get_depth_from_gebco(float(lons[i]), float(lats[i])) for i in range(n_sites)], dtype=np.float32)
    depth_da = xr.DataArray(
        np.broadcast_to(depth_values[np.newaxis, :], (n_cyclones, n_sites)),
        dims=["time", "site"],
        coords={"time": spectra.time, "site": spectra.site},
        name="dpt",
    )
    spectra = spectra.assign(dpt=depth_da)

    # Expand wind to (time, site) for PTM1 - use same coords as dpt so check_same_coordinates passes
    dpt_coords = dict(spectra.dpt.coords)
    wspd_exp = xr.DataArray(
        np.broadcast_to(wspd.values[:, np.newaxis], (n_cyclones, n_sites)),
        dims=["time", "site"],
        coords=dpt_coords,
        name="wspd",
    )
    wdir_exp = xr.DataArray(
        np.broadcast_to(wdir.values[:, np.newaxis], (n_cyclones, n_sites)),
        dims=["time", "site"],
        coords=dpt_coords,
        name="wdir",
    )

    # 1. Full spectrum (non-partitioned): hs, tp, tm02, dp, dm
    print("  Computing full spectrum parameters (hs, tp, tm02, dp, dm)...")
    spec_full = spectra.spec
    full_vars = {}
    for out_name, method_name, long_name, units in FULL_SPECTRUM_OUTPUTS:
        method = getattr(spec_full, method_name)
        da = method()
        if hasattr(da, "load") and da.chunks:
            da = da.load()
        full_vars[out_name] = da

    # 2. PTM1 partitioning (swells=1 -> 2 parts)
    print("  Running PTM1 partitioning (swells=1, 2 parts)...")
    dspart = spectra.spec.partition.ptm1(wspd_exp, wdir_exp, spectra.dpt, swells=1, smooth=False)
    if hasattr(dspart, "chunks") and dspart.chunks:
        dspart = dspart.load()

    n_parts = min(2, dspart.sizes.get("part", 2))
    part_indices = list(range(n_parts))

    phs_list, ptp_list, pdp_list, spr_list = [], [], [], []
    for p in part_indices:
        part_ds = xr.Dataset({"efth": dspart.isel(part=p)})
        spec_part = part_ds.spec
        phs_list.append(spec_part.hs())
        ptp_list.append(spec_part.tp())
        pdp_list.append(spec_part.dpm())
        spr_list.append(spec_part.dspr())

    site_coords = spectra.site.values if hasattr(spectra.site, "values") else np.arange(n_sites)
    site_lat = xr.DataArray(np.atleast_1d(lats).flatten()[:n_sites], dims=["site"], coords={"site": site_coords})
    site_lon = xr.DataArray(np.atleast_1d(lons).flatten()[:n_sites], dims=["site"], coords={"site": site_coords})

    def _save(ds_out, fname):
        ds_out = ds_out.assign_coords(lat=site_lat, lon=site_lon)
        ds_out.to_netcdf(fname, encoding={v: {"zlib": True, "complevel": 4} for v in ds_out.data_vars})

    # Save full spectrum vars
    for out_name, _, long_name, units in FULL_SPECTRUM_OUTPUTS:
        da = full_vars[out_name]
        ds_out = xr.Dataset({out_name: da})
        ds_out[out_name].attrs.update({"long_name": long_name, "units": units})
        fname = os.path.join(OUTPUT_DIR, f"{out_name}_{grid_name}_cyclones.nc")
        _save(ds_out, fname)
        print(f"  Saved {fname} ({n_sites} sites)")

    # Save partitioned vars as phs0, phs1, ptp0, ptp1, pdp0, pdp1, spr0, spr1
    part_meta = {"phs": ("partition_significant_wave_height", "m"), "ptp": ("partition_peak_period", "s"),
                 "pdp": ("partition_peak_direction", "degrees"), "spr": ("partition_directional_spreading", "degrees")}
    for i, (phs_p, ptp_p, pdp_p, spr_p) in enumerate(zip(phs_list, ptp_list, pdp_list, spr_list)):
        for pfx, da, (ln, u) in [("phs", phs_p, part_meta["phs"]), ("ptp", ptp_p, part_meta["ptp"]),
                                  ("pdp", pdp_p, part_meta["pdp"]), ("spr", spr_p, part_meta["spr"])]:
            varname = f"{pfx}{i}"
            ds_out = xr.Dataset({varname: da})
            ds_out[varname].attrs.update({"long_name": ln, "units": u})
            fname = os.path.join(OUTPUT_DIR, f"{varname}_{grid_name}_cyclones.nc")
            _save(ds_out, fname)
        print(f"  Saved partition {i} (phs{i}, ptp{i}, pdp{i}, spr{i})")

    ds.close()
    return {"label": grid_name}

In [ ]:
def process_grid_cyclones(grid_name, max_cyclones=None):
    """Partition reconstructed spectra for one grid, processing one cyclone file at a time.
    Outputs are saved per cyclone id to avoid creating giant merged inputs.

    Parameters
    ----------
    grid_name : str
        Grid label (e.g., 'grid1').
    max_cyclones : int | None
        If provided, only process the first N cyclone files (sorted by id).
    """
    import glob
    import re

    spectra_glob = os.path.join(
        BASE_DIR, f"{grid_name}", "outputs", f"reconstructed_spectra_{grid_name}_cyclone_*.nc"
    )
    rx = re.compile(r".*_cyclone_(\d+)\.nc$")

    spectra_files = []
    for p in glob.glob(spectra_glob):
        m = rx.match(p)
        if m:
            spectra_files.append((int(m.group(1)), p))

    spectra_files.sort(key=lambda x: x[0])
    if max_cyclones is not None:
        spectra_files = spectra_files[:max_cyclones]

    if not spectra_files:
        raise FileNotFoundError(
            f"No per-cyclone reconstructed spectra found for {grid_name}. "
            f"Expected files like: {spectra_glob}"
        )

    print(f"\n{'='*60}")
    print(f"Processing {grid_name} with {len(spectra_files)} cyclone files")
    print(f"{'='*60}")

    point_name = GRID_TO_POINT[grid_name]
    point_dim = "point" if "point" in wind_ds.dims else next((d for d in wind_ds.dims if "point" in d.lower()), None)
    if point_dim is None:
        raise ValueError("Wind data must have a point-like dimension")

    pt_idx = point_name.split("_")[-1] if "_" in point_name else point_name
    try:
        pt_idx = int(pt_idx) - 1
    except ValueError:
        pass

    try:
        vx_point = wind_ds["Windv_x"].sel(**{point_dim: point_name}, method="nearest").values
        vy_point = wind_ds["Windv_y"].sel(**{point_dim: point_name}, method="nearest").values
    except Exception:
        vx_point = wind_ds["Windv_x"].isel(**{point_dim: pt_idx}).values
        vy_point = wind_ds["Windv_y"].isel(**{point_dim: pt_idx}).values

    vx_point = np.asarray(vx_point)
    vy_point = np.asarray(vy_point)

    cached_depth_values = None
    cached_site_lat = None
    cached_site_lon = None
    processed = 0

    def _wind_for_cyclone(cyclone_idx, n_steps, time_coord):
        if vx_point.ndim >= 2 and cyclone_idx < vx_point.shape[0]:
            vx = np.asarray(vx_point[cyclone_idx]).reshape(-1)
            vy = np.asarray(vy_point[cyclone_idx]).reshape(-1)
        else:
            vx_flat = vx_point.reshape(-1)
            vy_flat = vy_point.reshape(-1)
            start = cyclone_idx * n_steps
            end = start + n_steps
            if end <= len(vx_flat):
                vx = vx_flat[start:end]
                vy = vy_flat[start:end]
            else:
                vx = vx_flat[:n_steps]
                vy = vy_flat[:n_steps]

        if len(vx) < n_steps:
            pad = n_steps - len(vx)
            vx = np.pad(vx, (0, pad), mode="edge")
            vy = np.pad(vy, (0, pad), mode="edge")
        else:
            vx = vx[:n_steps]
            vy = vy[:n_steps]

        wspd = np.sqrt(vx**2 + vy**2)
        wdir = (270 - np.degrees(np.arctan2(vy, vx))) % 360.0
        wspd_da = xr.DataArray(wspd, dims=["time"], coords={"time": time_coord})
        wdir_da = xr.DataArray(wdir, dims=["time"], coords={"time": time_coord})
        return wspd_da, wdir_da

    def _save(ds_out, fname):
        ds_out = ds_out.assign_coords(lat=cached_site_lat, lon=cached_site_lon)
        encoding = {v: {"zlib": True, "complevel": 4} for v in ds_out.data_vars}
        ds_out.to_netcdf(fname, encoding=encoding)

    part_meta = {
        "phs": ("partition_significant_wave_height", "m"),
        "ptp": ("partition_peak_period", "s"),
        "pdp": ("partition_peak_direction", "degrees"),
        "spr": ("partition_directional_spreading", "degrees"),
    }

    for cyclone_idx, spectra_path in spectra_files:
        ds = xr.open_dataset(spectra_path)
        try:
            efth_var = "efth" if "efth" in ds.data_vars else list(ds.data_vars)[0]
            spectra = ds.rename({efth_var: "efth"})

            if "cyclone_id" in spectra.dims and spectra.sizes["cyclone_id"] == 1:
                spectra = spectra.isel(cyclone_id=0, drop=True)

            n_steps = spectra.sizes["time"]
            n_sites = spectra.sizes["site"]

            if "coord_x" in spectra.coords and "coord_y" in spectra.coords:
                lons = spectra.coord_x.values
                lats = spectra.coord_y.values
            elif "lon" in spectra.coords and "lat" in spectra.coords:
                lons, lats = spectra.lon.values, spectra.lat.values
            else:
                raise ValueError("Spectra must have coord_x/coord_y or lon/lat")

            if cached_depth_values is None:
                cached_depth_values = np.array(
                    [get_depth_from_gebco(float(lons[i]), float(lats[i])) for i in range(n_sites)],
                    dtype=np.float32,
                )
                site_coords = spectra.site.values if hasattr(spectra.site, "values") else np.arange(n_sites)
                cached_site_lat = xr.DataArray(
                    np.atleast_1d(lats).flatten()[:n_sites],
                    dims=["site"],
                    coords={"site": site_coords},
                )
                cached_site_lon = xr.DataArray(
                    np.atleast_1d(lons).flatten()[:n_sites],
                    dims=["site"],
                    coords={"site": site_coords},
                )

            depth_da = xr.DataArray(
                np.broadcast_to(cached_depth_values[np.newaxis, :], (n_steps, n_sites)),
                dims=["time", "site"],
                coords={"time": spectra.time, "site": spectra.site},
                name="dpt",
            )
            spectra = spectra.assign(dpt=depth_da)

            wspd, wdir = _wind_for_cyclone(cyclone_idx, n_steps, spectra.time)
            dpt_coords = dict(spectra.dpt.coords)
            wspd_exp = xr.DataArray(
                np.broadcast_to(wspd.values[:, np.newaxis], (n_steps, n_sites)),
                dims=["time", "site"],
                coords=dpt_coords,
                name="wspd",
            )
            wdir_exp = xr.DataArray(
                np.broadcast_to(wdir.values[:, np.newaxis], (n_steps, n_sites)),
                dims=["time", "site"],
                coords=dpt_coords,
                name="wdir",
            )

            spec_full = spectra.spec
            full_vars = {}
            for out_name, method_name, _, _ in FULL_SPECTRUM_OUTPUTS:
                method = getattr(spec_full, method_name)
                da = method()
                if hasattr(da, "load") and da.chunks:
                    da = da.load()
                full_vars[out_name] = da

            dspart = spectra.spec.partition.ptm1(wspd_exp, wdir_exp, spectra.dpt, swells=1, smooth=False)
            if hasattr(dspart, "chunks") and dspart.chunks:
                dspart = dspart.load()

            n_parts = min(2, dspart.sizes.get("part", 2))
            phs_list, ptp_list, pdp_list, spr_list = [], [], [], []
            for p in range(n_parts):
                part_ds = xr.Dataset({"efth": dspart.isel(part=p)})
                spec_part = part_ds.spec
                phs_list.append(spec_part.hs())
                ptp_list.append(spec_part.tp())
                pdp_list.append(spec_part.dpm())
                spr_list.append(spec_part.dspr())

            for out_name, _, long_name, units in FULL_SPECTRUM_OUTPUTS:
                da = full_vars[out_name]
                ds_out = xr.Dataset({out_name: da})
                ds_out[out_name].attrs.update({"long_name": long_name, "units": units})
                fname = os.path.join(OUTPUT_DIR, f"{out_name}_{grid_name}_cyclone_{cyclone_idx}.nc")
                _save(ds_out, fname)

            for i, (phs_p, ptp_p, pdp_p, spr_p) in enumerate(zip(phs_list, ptp_list, pdp_list, spr_list)):
                for pfx, da, (ln, u) in [
                    ("phs", phs_p, part_meta["phs"]),
                    ("ptp", ptp_p, part_meta["ptp"]),
                    ("pdp", pdp_p, part_meta["pdp"]),
                    ("spr", spr_p, part_meta["spr"]),
                ]:
                    varname = f"{pfx}{i}"
                    ds_out = xr.Dataset({varname: da})
                    ds_out[varname].attrs.update({"long_name": ln, "units": u})
                    fname = os.path.join(OUTPUT_DIR, f"{varname}_{grid_name}_cyclone_{cyclone_idx}.nc")
                    _save(ds_out, fname)

            processed += 1
            if processed % 10 == 0 or processed == len(spectra_files):
                print(f"  Processed {processed}/{len(spectra_files)} cyclones for {grid_name}")

        finally:
            ds.close()

    return {"label": grid_name, "n_cyclones": processed}


In [ ]:
grid_order = sorted(TARGET_POINTS.keys())
per_grid_results = {}

# Use 10 for quick validation; set to None for full run.
MAX_CYCLONES_PER_GRID = 10

for grid_name in grid_order:
    try:
        per_grid_results[grid_name] = process_grid_cyclones(
            grid_name,
            max_cyclones=MAX_CYCLONES_PER_GRID,
        )
    except Exception as e:
        print(f"Error processing {grid_name}: {e}")
        raise

In [ ]:
import sys
import os
BASE_DIR = os.path.abspath(".")
# Add NorthCarolina_Emulator to path for utils (crop function)
NC_EMULATOR = os.path.join(os.path.dirname(BASE_DIR), "Cyclones_NC")
if NC_EMULATOR not in sys.path:
    sys.path.insert(0, NC_EMULATOR)
from utils.postprocessing_all_grids import crop_concatenated_files_by_spatial_mask

# Use our partitions output (from previous cells)
input_dir = os.path.join(BASE_DIR, "outputs", "partitions_cyclones")
output_dir = os.path.join(BASE_DIR, "outputs", "cropped_variables")
shoreline_file = os.path.join(BASE_DIR, "inputs", "CoastSat_shoreline_NC_merged.geojson")
if not os.path.exists(shoreline_file):
    shoreline_file = os.path.join(os.path.dirname(BASE_DIR), "inputs", "CoastSat_shoreline_NC_merged.geojson")

# Define buoys to preserve
buoys = {
    # Southern buoys
    'SSBN7': (-78.484, 33.838),
    '41119': (-78.483, 33.842),
    'OCPN7': (-78.147, 33.911),
    '41108': (-78.016, 33.721), 
    '41013': (-77.764, 33.441),
    '41110': (-77.715, 34.142),
    '41109': (-77.300, 34.484),
    '41035': (-77.281, 34.476),
    '41036': (-76.949, 34.207),
    '41159': (-76.944, 34.211),
    '41007': (-76.5, 34.2),
    
    # Central buoys
    'jprn7': (-75.5870, 35.9120),
    '41017': (-75.1000, 35.4000),
    '41015': (-75.3000, 35.4000),
    '41120': (-75.2580, 35.2580),
    'dsln7': (-75.2970, 35.1530),
    '41025': (-75.4540, 35.0100),
    '44095': (-75.3300, 35.7500),
    '44086': (-75.3300, 35.7500),
    
    # Northern buoys
    '44006': (-75.4000, 36.3000),
    '44079': (-75.5930, 36.1750),
    '44056': (-75.7140, 36.2000),
    '44100': (-75.5930, 36.2580),
    '44019': (-75.2000, 36.4000),
}



# Variables: full spectrum (hs, tp, tm02, dp, dm) + partitioned 2 parts (phs0/1, ptp0/1, pdp0/1, spr0/1)
variables = [
    'hs', 'tp', 'tm02', 'dp', 'dm',
    'phs0', 'phs1', 'ptp0', 'ptp1', 'pdp0', 'pdp1', 'spr0', 'spr1',
]

crop_concatenated_files_by_spatial_mask(
    input_dir=input_dir,
    output_dir=output_dir,
    shoreline_geojson_file=shoreline_file,
    grid=['grid1', 'grid2', 'grid3', 'grid4'],
    variable=variables,
    buoy_coordinates=buoys,
)

In [ ]:
import sys
import os
import re
import glob
import shutil
from pathlib import Path

BASE_DIR = os.path.abspath(".")
NC_EMULATOR = os.path.join(os.path.dirname(BASE_DIR), "Cyclones_NC")
if NC_EMULATOR not in sys.path:
    sys.path.insert(0, NC_EMULATOR)
from utils.postprocessing_all_grids import crop_concatenated_files_by_spatial_mask

# Full partitions output (generated in previous cells)
full_input_dir = os.path.join(BASE_DIR, "outputs", "partitions_cyclones")

# Test subset directories: first N cyclone ids per grid
TEST_FIRST_N = 10
input_dir = os.path.join(BASE_DIR, "outputs", f"partitions_cyclones_first{TEST_FIRST_N}")
output_dir = os.path.join(BASE_DIR, "outputs", f"cropped_variables_first{TEST_FIRST_N}")

shoreline_file = os.path.join(BASE_DIR, "inputs", "CoastSat_shoreline_NC_merged.geojson")
if not os.path.exists(shoreline_file):
    shoreline_file = os.path.join(os.path.dirname(BASE_DIR), "inputs", "CoastSat_shoreline_NC_merged.geojson")

buoys = {
    'SSBN7': (-78.484, 33.838),
    '41119': (-78.483, 33.842),
    'OCPN7': (-78.147, 33.911),
    '41108': (-78.016, 33.721),
    '41013': (-77.764, 33.441),
    '41110': (-77.715, 34.142),
    '41109': (-77.300, 34.484),
    '41035': (-77.281, 34.476),
    '41036': (-76.949, 34.207),
    '41159': (-76.944, 34.211),
    '41007': (-76.5, 34.2),
    'jprn7': (-75.5870, 35.9120),
    '41017': (-75.1000, 35.4000),
    '41015': (-75.3000, 35.4000),
    '41120': (-75.2580, 35.2580),
    'dsln7': (-75.2970, 35.1530),
    '41025': (-75.4540, 35.0100),
    '44095': (-75.3300, 35.7500),
    '44086': (-75.3300, 35.7500),
    '44006': (-75.4000, 36.3000),
    '44079': (-75.5930, 36.1750),
    '44056': (-75.7140, 36.2000),
    '44100': (-75.5930, 36.2580),
    '44019': (-75.2000, 36.4000),
}

variables = [
    'hs', 'tp', 'tm02', 'dp', 'dm',
    'phs0', 'phs1', 'ptp0', 'ptp1', 'pdp0', 'pdp1', 'spr0', 'spr1',
]

grid_names = ['grid1', 'grid2', 'grid3', 'grid4']

# Build a lightweight test input folder with only first N cyclone ids per grid.
input_dir_path = Path(input_dir)
if input_dir_path.exists():
    shutil.rmtree(input_dir_path)
input_dir_path.mkdir(parents=True, exist_ok=True)

id_pattern = re.compile(r"^(?P<var>[a-z0-9]+)_(?P<grid>grid\d+)_cyclone_(?P<id>\d+)\.nc$")
selected_ids_by_grid = {}

for grid_name in grid_names:
    hs_pattern = os.path.join(full_input_dir, f"hs_{grid_name}_cyclone_*.nc")
    hs_files = sorted(glob.glob(hs_pattern))
    ids = []
    for p in hs_files:
        m = id_pattern.match(os.path.basename(p))
        if m:
            ids.append(int(m.group("id")))
    selected_ids_by_grid[grid_name] = set(ids[:TEST_FIRST_N])

linked = 0
for src in Path(full_input_dir).glob("*.nc"):
    m = id_pattern.match(src.name)
    if not m:
        continue

    var = m.group("var")
    grid_name = m.group("grid")
    cyclone_id = int(m.group("id"))

    if var not in variables:
        continue
    if grid_name not in selected_ids_by_grid:
        continue
    if cyclone_id not in selected_ids_by_grid[grid_name]:
        continue

    dst = input_dir_path / src.name
    try:
        os.symlink(src, dst)
    except OSError:
        shutil.copy2(src, dst)
    linked += 1

print(f"Prepared test crop input dir: {input_dir}")
print(f"Linked/copied {linked} files for first {TEST_FIRST_N} cyclone ids per grid")

crop_concatenated_files_by_spatial_mask(
    input_dir=input_dir,
    output_dir=output_dir,
    shoreline_geojson_file=shoreline_file,
    grid=grid_names,
    variable=variables,
    buoy_coordinates=buoys,
)


In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import gc  # For garbage collection
from pathlib import Path
from utils.outputs_grids import merge_multiple_grids

import os

GRIDS = {
    "grid1": "grid1",
    "grid2": "grid2",
    "grid3": "grid3",
    "grid4": "grid4",
}

def validate_netcdf_file(file_path, var_name=None):
    """
    Validate that a NetCDF file can be opened and read.
    Returns (is_valid, error_message)
    """
    try:
        with xr.open_dataset(file_path) as ds:
            # Try to access basic info
            _ = ds.sizes
            # If var_name specified, try to access it
            if var_name and var_name in ds.data_vars:
                # Try to read a small sample (first site, first time)
                sample = ds[var_name].isel(site=0, time=0).values
                _ = sample
        return True, None
    except Exception as e:
        return False, str(e)

def cleanup_corrupted_temp_files(directory, var_name=None):
    """
    Find and delete corrupted temporary NetCDF files.
    Returns count of deleted files.
    """
    deleted_count = 0
    temp_files = list(directory.glob("temp_*.nc"))
    
    if len(temp_files) == 0:
        return 0
    
    print(f"\n{'='*80}")
    print(f"Checking for corrupted temporary files...")
    print(f"{'='*80}")
    print(f"Found {len(temp_files)} temporary file(s) to check")
    
    for temp_file in temp_files:
        is_valid, error = validate_netcdf_file(temp_file, var_name)
        if not is_valid:
            print(f"  ✗ Corrupted: {temp_file.name}")
            print(f"    Error: {error}")
            try:
                temp_file.unlink()
                deleted_count += 1
                print(f"    ✓ Deleted")
            except Exception as e:
                print(f"    ⚠ Could not delete: {e}")
        else:
            print(f"  ✓ Valid: {temp_file.name}")
    
    if deleted_count > 0:
        print(f"\n  Cleaned up {deleted_count} corrupted temporary file(s)")
    else:
        print(f"\n  All temporary files are valid")
    print(f"{'='*80}\n")
    
    return deleted_count

# ============================================================================
# MERGE ALL GRIDS SETTINGS
# ============================================================================
# Convert both to Path objects
BASE_DIR = Path(os.path.abspath("."))
CONCATENATED_DIR = BASE_DIR / "outputs" / "cropped_variables"
FINAL_MERGED_DIR = BASE_DIR / "outputs" / "merged_grids"

# Now these will work
CONCATENATED_DIR.mkdir(parents=True, exist_ok=True)
FINAL_MERGED_DIR.mkdir(parents=True, exist_ok=True)

# And this will work too
if not CONCATENATED_DIR.exists():
    raise ValueError(f"Concatenated grids directory not found: {CONCATENATED_DIR}")

# Blending parameters (same as verification)
BLEND_STEEPNESS = 5
TOLERANCE_DEG = 0.001

# ============================================================================
# VARIABLES TO PROCESS - choose one option:
# ============================================================================
# Option A: Process ALL discovered variables (auto-discover from concatenated files)
VARIABLES_TO_PROCESS = ['hs', 'tp', 'tm02', 'dp', 'dm']

# Option B: Process ONLY these variables (set to a list; overrides auto-discovery)
# VARIABLES_TO_PROCESS =   # full spectrum only
# VARIABLES_TO_PROCESS = ['hs', 'tp']  # minimal subset

# Option C: When using Option A, exclude specific variables from discovery
VARIABLES_TO_EXCLUDE = []  # e.g. ['spr0', 'spr1'] to skip spread, or [] for all

# Grid names in order (will merge sequentially: grid1+grid2, then result+grid3, then result+grid4)
GRID_NAMES = list(GRIDS.keys())  # ['grid1', 'grid2', 'grid3', 'grid4']
# ============================================================================
print(f"{'='*80}")
print(f"MERGING ALL GRIDS USING CONCATENATED FILES")
print(f"{'='*80}")
print(f"Concatenated files directory: {CONCATENATED_DIR}")
print(f"Output directory: {FINAL_MERGED_DIR}")
print(f"Blend steepness: {BLEND_STEEPNESS}")
print(f"Tolerance: {TOLERANCE_DEG} degrees")
print(f"Grids to merge (in order): {', '.join(GRID_NAMES)}")
print(f"{'='*80}")

# Check if concatenated directory exists
if not CONCATENATED_DIR.exists():
    raise ValueError(f"Concatenated grids directory not found: {CONCATENATED_DIR}")

# Discover variables if not specified
if VARIABLES_TO_PROCESS is None:
    print(f"\nDiscovering variables from concatenated files...")
    variables_found = set()
    for grid_name in GRID_NAMES:
        for file in CONCATENATED_DIR.glob(f"*_{grid_name}_cyclones_masked.nc"):
            # Extract variable name (e.g., hs_grid1_cyclones_masked.nc -> hs)
            stem = file.stem.replace('_masked', '').replace('_cyclones', '')  # hs_grid1
            var_name = stem.rsplit(f'_{grid_name}', 1)[0]
            variables_found.add(var_name)
    
    # Apply exclusions (if VARIABLES_TO_EXCLUDE is set)
    if VARIABLES_TO_EXCLUDE:
        variables_found = variables_found - set(VARIABLES_TO_EXCLUDE)
        print(f"  Excluded: {', '.join(VARIABLES_TO_EXCLUDE)}")
    
    # Define priority order for variables (sea states: full spectrum + partitioned)
    PRIORITY_VARIABLES = ['hs', 'tp', 'tm02', 'dp', 'dm']  # Full spectrum first
    
    # Sort variables: priority variables first (in specified order), then rest alphabetically
    priority_vars = []
    other_vars = []
    
    for var in PRIORITY_VARIABLES:
        if var in variables_found:
            priority_vars.append(var)
    
    for var in sorted(variables_found):
        if var not in PRIORITY_VARIABLES:
            other_vars.append(var)
    
    VARIABLES_TO_PROCESS = priority_vars + other_vars
    print(f"  Found variables: {', '.join(VARIABLES_TO_PROCESS)}")
    print(f"  Processing order: Priority variables first ({', '.join(priority_vars)}), then others alphabetically")
else:
    print(f"\nUsing specified variables: {', '.join(VARIABLES_TO_PROCESS)}")

if len(VARIABLES_TO_PROCESS) == 0:
    raise ValueError("No variables found to process!")

# Check which variables have already been processed
print(f"\n{'='*80}")
print(f"Checking for already processed variables...")
print(f"{'='*80}")
already_processed = []
to_process = []

for var_name in VARIABLES_TO_PROCESS:
    final_output_file = FINAL_MERGED_DIR / f"{var_name}_merged_all.nc"
    if final_output_file.exists():
        already_processed.append(var_name)
        print(f"  ✓ {var_name}: Already processed ({final_output_file.name})")
    else:
        to_process.append(var_name)

if len(already_processed) > 0:
    print(f"\n  Skipping {len(already_processed)} already processed variable(s): {', '.join(already_processed)}")

if len(to_process) == 0:
    print(f"\n{'='*80}")
    print(f"All variables have already been processed!")
    print(f"{'='*80}")
    print(f"\nFinal merged files are in: {FINAL_MERGED_DIR}")
    raise SystemExit("No variables to process. All done!")

print(f"\n  Will process {len(to_process)} variable(s): {', '.join(to_process)}")
print(f"{'='*80}")

# Clean up any corrupted temp files before starting
print(f"\n{'='*80}")
print(f"CLEANING UP CORRUPTED TEMPORARY FILES")
print(f"{'='*80}")
cleanup_corrupted_temp_files(FINAL_MERGED_DIR)

# Process each variable
for var_idx, var_name in enumerate(to_process):
    print(f"\n{'='*80}")
    print(f"Processing variable: {var_name} ({var_idx+1}/{len(to_process)})")
    print(f"{'='*80}")
    
    # Output file for final merged result
    final_output_file = FINAL_MERGED_DIR / f"{var_name}_merged_all.nc"
    
    # Double-check if already processed (safety check)
    if final_output_file.exists():
        print(f"  ⏭ Skipping {var_name}: Final merged file already exists")
        print(f"     {final_output_file}")
        continue
    
    # Collect concatenated files for all grids
    grid_files = []
    missing_grids = []
    
    for grid_name in GRID_NAMES:
        concat_file = CONCATENATED_DIR / f"{var_name}_{grid_name}_cyclones_masked.nc"
        if concat_file.exists():
            grid_files.append(concat_file)
            print(f"  ✓ Found {grid_name}: {concat_file.name}")
        else:
            missing_grids.append(grid_name)
            print(f"  ✗ Missing {grid_name}: {concat_file}")
    
    if len(grid_files) == 0:
        print(f"  ⚠ No files found for {var_name}, skipping...")
        continue
    
    if len(missing_grids) > 0:
        print(f"  ⚠ Warning: {len(missing_grids)} grid(s) missing: {', '.join(missing_grids)}")
        print(f"  Continuing with {len(grid_files)} available grid(s)")
    
    # Initialize variables for cleanup
    current_merged = None
    ds_grid = None
    ds_grid_aligned = None
    
    try:
        # Load first file to detect actual variable name (with proper cleanup)
        with xr.open_dataset(grid_files[0]) as ds_sample:
            data_vars = [v for v in ds_sample.data_vars if v not in ['lon', 'lat', 'coord_x', 'coord_y', 'site_id', 'polygon_lon', 'polygon_lat']]
            actual_var_name = var_name if var_name in data_vars else data_vars[0] if data_vars else var_name
        
        print(f"\n  Actual variable name: {actual_var_name}")
        print(f"  Merging {len(grid_files)} grids sequentially...")
        
        # Find common time index (sea states: integer 0-48; datetime for time-series)
        print(f"\n  Finding time ranges across all grids...")
        all_times = []
        for grid_file in grid_files:
            with xr.open_dataset(grid_file) as ds_temp:
                tvals = ds_temp.time.values
                if np.issubdtype(tvals.dtype, np.integer):
                    times = np.asarray(tvals)
                else:
                    times = pd.to_datetime(tvals)
                all_times.extend(times)
                print(f"    {grid_file.name}: {times[0]} to {times[-1]} ({len(times)} timesteps)")
        
        all_times = np.unique(np.sort(np.asarray(all_times).ravel()))
        if np.issubdtype(all_times.dtype, np.integer):
            common_time = xr.DataArray(all_times, dims=['time'], name='time')
        else:
            common_time = xr.DataArray(pd.to_datetime(sorted(set(all_times))), dims=['time'], name='time')
        print(f"\n  Common time: {len(common_time)} unique timesteps")
        
        # Sequential merging: merge grids one by one
        # Check if we need to resume from a previous step
        # Look for the highest step number with existing temp files (both merged and grid temp files)
        # Also validate that the files are not corrupted
        resume_from_step = None
        for step in range(len(grid_files) - 1, 0, -1):  # Check steps 1 to len-1 (step 0 doesn't create temp files)
            temp_merged_file = FINAL_MERGED_DIR / f"temp_{var_name}_merged_step{step}.nc"
            # Extract grid name (e.g., hs_grid1_cyclones_masked.nc -> grid1)
            stem_no_masked = grid_files[step].stem.replace('_masked', '').replace('_cyclones', '')
            grid_name_at_step = stem_no_masked.rsplit('_', 1)[-1]
            temp_grid_file = FINAL_MERGED_DIR / f"temp_{var_name}_{grid_name_at_step}_step{step}.nc"
            if temp_merged_file.exists() and temp_grid_file.exists():
                # Validate both files before using them
                is_valid_merged, error_merged = validate_netcdf_file(temp_merged_file, actual_var_name)
                is_valid_grid, error_grid = validate_netcdf_file(temp_grid_file, actual_var_name)
                
                if is_valid_merged and is_valid_grid:
                    resume_from_step = step
                    print(f"\n  ↻ Detected resume point: Step {step + 1} (found valid temporary files)")
                    print(f"     Will resume from: {temp_merged_file.name} and {temp_grid_file.name}")
                    break
                else:
                    # Files are corrupted, delete them
                    print(f"\n  ⚠ Detected corrupted temporary files at step {step + 1}, deleting...")
                    if not is_valid_merged:
                        print(f"     {temp_merged_file.name}: {error_merged}")
                        try:
                            temp_merged_file.unlink()
                        except:
                            pass
                    if not is_valid_grid:
                        print(f"     {temp_grid_file.name}: {error_grid}")
                        try:
                            temp_grid_file.unlink()
                        except:
                            pass
                    print(f"     Will recreate files from scratch")
        
        for merge_step, grid_file in enumerate(grid_files):
            # Extract grid name (e.g., hs_grid1_cyclones_masked.nc -> grid1)
            stem_no_masked = grid_file.stem.replace('_masked', '').replace('_cyclones', '')
            grid_name = stem_no_masked.rsplit('_', 1)[-1]
            
            # If resuming, skip steps before the resume point
            if resume_from_step is not None and merge_step < resume_from_step:
                print(f"\n    Step {merge_step + 1}: Skipping (already completed)")
                continue
            
            # Ensure previous datasets are closed before proceeding
            if ds_grid is not None:
                try:
                    ds_grid.close()
                except:
                    pass
                ds_grid = None
            if ds_grid_aligned is not None:
                try:
                    ds_grid_aligned.close()
                except:
                    pass
                ds_grid_aligned = None
            
            # If this is the resume step, load current_merged from temp file
            temp_next_file = FINAL_MERGED_DIR / f"temp_{var_name}_{grid_name}_step{merge_step}.nc"
            is_resuming = resume_from_step is not None and merge_step == resume_from_step and temp_next_file.exists()
            
            if is_resuming:
                temp_merged_file = FINAL_MERGED_DIR / f"temp_{var_name}_merged_step{resume_from_step}.nc"
                if temp_merged_file.exists():
                    print(f"\n    Step {merge_step + 1}: Resuming from checkpoint...")
                    # Close any existing current_merged before loading new one
                    if current_merged is not None:
                        try:
                            current_merged.close()
                        except:
                            pass
                    current_merged = xr.open_dataset(temp_merged_file)
                    print(f"      ↻ Loaded previous result: {current_merged.sizes['site']} sites, {current_merged.sizes['time']} timesteps")
            else:
                # Load grid and reindex to common time index (handles gaps by filling with NaN)
                print(f"\n    Step {merge_step + 1}: Loading {grid_name}...")
                ds_grid = xr.open_dataset(grid_file)
                
                # Reindex to common time (missing times will be NaN)
                ds_grid_aligned = ds_grid.reindex(time=common_time, method=None)
                
                # Check for gaps
                times_grid = ds_grid.time.values
                missing_times = set(np.asarray(all_times).ravel()) - set(np.asarray(times_grid).ravel())
                if len(missing_times) > 0:
                    print(f"      ⚠ Note: {len(missing_times)} timesteps missing in {grid_name} (will be NaN in merged result)")
            
            if merge_step == 0:
                # First grid: use aligned dataset
                print(f"      Aligning to common time index...")
                current_merged = ds_grid_aligned
                print(f"      ✓ Loaded {grid_name}: {current_merged.sizes['site']} sites, {current_merged.sizes['time']} timesteps")
                
                # Filter out unwanted variables from first grid
                vars_to_drop = []
                for var in current_merged.data_vars:
                    if var in ['coord_x', 'coord_y', 'site_id'] or var.startswith('weight_'):
                        vars_to_drop.append(var)
                
                if vars_to_drop:
                    current_merged = current_merged.drop_vars(vars_to_drop)
                    print(f"      Removed extra variables: {', '.join(vars_to_drop)}")
                
                # Clear attributes
                current_merged.attrs = {}
                    
                # Force garbage collection to free memory after merge
                collected = gc.collect()
                if collected > 0:
                    print(f"      Freed {collected} objects from memory")
                # Close original dataset (aligned version is now current_merged)
                ds_grid.close()
                ds_grid = None
                ds_grid_aligned = None  # Don't close, it's now current_merged
            else:
                # Merge current merged result with next grid
                print(f"      Preparing merge with {grid_name}...")
                
                # Check for existing temporary files (resume from checkpoint)
                temp_current_file = FINAL_MERGED_DIR / f"temp_{var_name}_merged_step{merge_step}.nc"
                temp_next_file = FINAL_MERGED_DIR / f"temp_{var_name}_{grid_name}_step{merge_step}.nc"
                
                # Check if we can resume from existing temp files (validate first)
                can_resume = False
                if temp_current_file.exists() and temp_next_file.exists():
                    # Validate both files
                    is_valid_current, error_current = validate_netcdf_file(temp_current_file, actual_var_name)
                    is_valid_next, error_next = validate_netcdf_file(temp_next_file, actual_var_name)
                    
                    if is_valid_current and is_valid_next:
                        can_resume = True
                        print(f"      ↻ Resuming from existing temporary files...")
                        print(f"         Found: {temp_current_file.name}")
                        print(f"         Found: {temp_next_file.name}")
                    else:
                        # Files are corrupted, delete them
                        print(f"      ⚠ Temporary files are corrupted, will recreate...")
                        if not is_valid_current:
                            print(f"         {temp_current_file.name}: {error_current}")
                            try:
                                temp_current_file.unlink()
                            except:
                                pass
                        if not is_valid_next:
                            print(f"         {temp_next_file.name}: {error_next}")
                            try:
                                temp_next_file.unlink()
                            except:
                                pass
                
                if can_resume:
                    # Close current_merged if it's open (it might be from previous step)
                    if current_merged is not None:
                        try:
                            current_merged.close()
                        except:
                            pass
                        current_merged = None
                else:
                    # Create temporary file for current merged result
                    if temp_current_file.exists():
                        print(f"      ↻ Found existing temp file: {temp_current_file.name}")
                        if current_merged is not None:
                            try:
                                current_merged.close()
                            except:
                                pass
                            current_merged = None
                    else:
                        print(f"      Saving current merged result to temp file...")
                        current_merged.to_netcdf(temp_current_file)
                        current_merged.close()
                        current_merged = None
                        gc.collect()  # Force garbage collection after saving
                    
                    # Create temporary file for aligned next grid
                    if temp_next_file.exists():
                        print(f"      ↻ Found existing temp file: {temp_next_file.name}")
                    else:
                        print(f"      Saving aligned grid to temp file...")
                        ds_grid_aligned.to_netcdf(temp_next_file)
                        gc.collect()  # Force garbage collection after saving
                    
                    # Close grid datasets now that we've saved them
                    if ds_grid is not None:
                        ds_grid.close()
                        ds_grid = None
                    if ds_grid_aligned is not None:
                        ds_grid_aligned.close()
                        ds_grid_aligned = None
                
                # Merge current merged result with next grid
                print(f"      Merging grids...")
                try:
                    current_merged = merge_multiple_grids(
                        grid_files=[temp_current_file, temp_next_file],
                        var_name=actual_var_name,
                        steepness=BLEND_STEEPNESS,
                        tolerance_deg=TOLERANCE_DEG,
                        output_file=None,  # Don't save intermediate files
                        use_quality_checks=False  # Disabled to reduce memory usage
                    )
                    print(f"      ✓ Merged result: {current_merged.sizes['site']} sites, {current_merged.sizes['time']} timesteps")
                    
                    # Filter out unwanted variables - keep only original input variables
                    # Remove: coord_x, coord_y, weight_* variables, site_id
                    vars_to_drop = []
                    for var in current_merged.data_vars:
                        if var in ['coord_x', 'coord_y', 'site_id'] or var.startswith('weight_'):
                            vars_to_drop.append(var)
                    
                    if vars_to_drop:
                        current_merged = current_merged.drop_vars(vars_to_drop)
                        print(f"      Removed extra variables: {', '.join(vars_to_drop)}")
                    
                    # Clear attributes that were added by merge_multiple_grids
                    current_merged.attrs = {}
                    
                    # Force garbage collection to free memory after merge
                    collected = gc.collect()
                    if collected > 0:
                        print(f"      Freed {collected} objects from memory")
                except (RuntimeError, OSError) as e:
                    error_msg = str(e)
                    if "HDF error" in error_msg or "NetCDF" in error_msg:
                        print(f"      ✗ HDF/NetCDF error during merge - temp files may be corrupted")
                        print(f"         Error: {error_msg}")
                        # Delete corrupted temp files and try to recreate from source
                        print(f"      Attempting to recreate temp files from source...")
                        try:
                            if temp_current_file.exists():
                                temp_current_file.unlink()
                            if temp_next_file.exists():
                                temp_next_file.unlink()
                        except:
                            pass
                        # Recreate temp files - need to reload from original sources
                        # This is a fallback - ideally we'd have the data in memory, but we'll reload
                        print(f"      Reloading data to recreate temp files...")
                        # Reload current merged from previous step or source
                        if merge_step > 0:
                            # Need to go back and recreate from step 0
                            raise RuntimeError(f"Cannot recover from corrupted temp files at step {merge_step + 1}. Please delete all temp files and restart from beginning.")
                    else:
                        # Re-raise if it's a different error
                        raise
                
                # Clean up temporary files immediately after merge
                if temp_current_file.exists():
                    try:
                        temp_current_file.unlink()
                    except:
                        pass
                if temp_next_file.exists():
                    try:
                        temp_next_file.unlink()
                    except:
                        pass
                
                # Force garbage collection after merge
                gc.collect()
        
        # Save final merged result
        print(f"\n  Saving final merged file...")
        if current_merged is not None:
            # Prepare encoding for compression (matching individual grid files)
            encoding = {}
            valid_encoding_keys = {
                "zlib", "complevel", "shuffle", "fletcher32", "contiguous",
                "chunksizes", "dtype", "_FillValue"
            }
            
            # Set encoding for all data variables (ensure float32 and compression)
            for var_name_enc in current_merged.data_vars:
                var = current_merged[var_name_enc]
                
                # Set encoding with compression
                var_enc = var.encoding.copy() if hasattr(var, 'encoding') and var.encoding else {}
                
                # Only set float32 encoding for numeric variables (not strings)
                if np.issubdtype(var.dtype, np.floating):
                    # Ensure float32 dtype
                    if var.dtype != np.float32:
                        current_merged[var_name_enc] = var.astype(np.float32)
                    # Set encoding with compression for numeric variables
                    var_enc["dtype"] = "float32"
                    var_enc["zlib"] = True
                    var_enc["complevel"] = 4  # Compression level (1-9, 4 is a good balance)
                    var_enc["shuffle"] = True  # Enable shuffle filter for better compression
                # For non-numeric variables (strings, etc.), don't set dtype or compression
                # They will use their default encoding
                
                # Clean encoding: exclude chunksizes to avoid None/int errors with netCDF4
                cleaned_enc = {k: v for k, v in var_enc.items() if k in valid_encoding_keys and k != "chunksizes"}
                encoding[var_name_enc] = cleaned_enc
            
            # Preserve coordinate encodings
            for coord_name in current_merged.coords:
                coord = current_merged[coord_name]
                coord_enc = coord.encoding.copy() if hasattr(coord, 'encoding') and coord.encoding else {}
                cleaned_coord_enc = {k: v for k, v in coord_enc.items() if k in valid_encoding_keys and k != "chunksizes"}
                encoding.setdefault(coord_name, cleaned_coord_enc)
            
            # Save with compression encoding using atomic write pattern
            # Write to temporary file first, then rename atomically to avoid permission issues
            import time
            temp_output_file = final_output_file.parent / f".{final_output_file.name}.tmp"
            max_retries = 3
            retry_delay = 1.0  # seconds
            
            try:
                # Clean up any existing temporary files first
                if temp_output_file.exists():
                    try:
                        temp_output_file.unlink()
                    except:
                        pass
                
                # Clean up any existing final file that might be locked
                if final_output_file.exists():
                    try:
                        # Try to close any open handles by forcing garbage collection
                        gc.collect()
                        time.sleep(0.5)  # Brief delay to allow file handles to close
                        final_output_file.unlink()
                        print(f"    ↻ Removed existing file: {final_output_file.name}")
                    except Exception as cleanup_error:
                        print(f"    ⚠ Could not remove existing file (may be locked): {cleanup_error}")
                        # Continue anyway - the temp file write should work
                
                # Write to temporary file
                for attempt in range(max_retries):
                    try:
                        current_merged.to_netcdf(temp_output_file, encoding=encoding)
                        break  # Success, exit retry loop
                    except (PermissionError, OSError) as e:
                        if attempt < max_retries - 1:
                            print(f"    ↻ Retry {attempt + 1}/{max_retries} after permission error...")
                            time.sleep(retry_delay * (attempt + 1))
                            gc.collect()  # Force cleanup before retry
                        else:
                            raise  # Re-raise on final attempt
                
                # Verify the temporary file was created successfully and has reasonable size
                if not temp_output_file.exists():
                    raise RuntimeError("Temporary file was not created after save")
                
                file_size_mb = temp_output_file.stat().st_size / (1024 * 1024)
                # Sea-state merged files (~49 timesteps) can be 50-150 KB; only reject truly empty (<10KB)
                if file_size_mb < 0.01:
                    temp_output_file.unlink()  # Clean up small file
                    raise RuntimeError(f"File size is suspiciously small: {file_size_mb:.1f} MB")
                
                # Atomically rename temporary file to final file
                # This is an atomic operation on most filesystems
                temp_output_file.replace(final_output_file)
                
                # Verify final file exists
                if not final_output_file.exists():
                    raise RuntimeError("Final file was not created after atomic rename")
                
                # Print summary
                print(f"\n  ✓ Successfully merged {len(grid_files)} grids for {var_name}")
                print(f"    Final output: {final_output_file}")
                print(f"    File size: {file_size_mb:.1f} MB (compressed, float32)")
                print(f"    Sites: {current_merged.sizes['site']}")
                print(f"    Timesteps: {current_merged.sizes['time']}")
                if 'time' in current_merged.coords:
                    tvals = current_merged.time.values
                    if np.issubdtype(tvals.dtype, np.integer):
                        print(f"    Time range: {tvals[0]} to {tvals[-1]} (sea states)")
                    else:
                        times = pd.to_datetime(tvals)
                        print(f"    Time range: {times[0]} to {times[-1]}")
                
            except Exception as save_error:
                # Clean up temporary file if it exists
                if temp_output_file.exists():
                    try:
                        temp_output_file.unlink()
                        print(f"    ↻ Cleaned up temporary file: {temp_output_file.name}")
                    except:
                        pass
                
                # Try to clean up final file if it exists and is corrupted
                if final_output_file.exists():
                    try:
                        # Check if file is suspiciously small (likely corrupted)
                        file_size_mb = final_output_file.stat().st_size / (1024 * 1024)
                        if file_size_mb < 0.01:  # <10KB = likely empty/corrupt
                            final_output_file.unlink()
                            print(f"    ⚠ Deleted corrupted file: {final_output_file.name}")
                    except:
                        pass
                
                # Re-raise the error so it gets caught by outer exception handler
                raise save_error
            finally:
                if current_merged is not None:
                    current_merged.close()
                    current_merged = None
        
        # Final cleanup
        gc.collect()
        
    except Exception as e:
        print(f"  ✗ Error processing {var_name}: {e}")
        import traceback
        traceback.print_exc()
        
        # Ensure cleanup on error
        if current_merged is not None:
            try:
                current_merged.close()
            except:
                pass
        if ds_grid is not None:
            try:
                ds_grid.close()
            except:
                pass
        if ds_grid_aligned is not None:
            try:
                ds_grid_aligned.close()
            except:
                pass
        
        gc.collect()
        continue

print(f"\n{'='*80}")
print(f"MERGING COMPLETE")
print(f"{'='*80}")
print(f"\nFinal merged files saved to: {FINAL_MERGED_DIR}")
print(f"One file per variable containing all grids merged sequentially.")


In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

try:
    from ipywidgets import interact, IntSlider
    USE_WIDGETS = True
except ImportError:
    USE_WIDGETS = False

# Load merged Hs (all grids blended)
try:
    _base = BASE_DIR if isinstance(BASE_DIR, str) else str(BASE_DIR)
except NameError:
    _base = os.path.abspath(".")
MERGED_DIR = os.path.join(_base, "outputs", "merged_grids")
hs_merged_path = os.path.join(MERGED_DIR, "hs_merged_all.nc")
if not os.path.exists(hs_merged_path):
    raise FileNotFoundError(f"Run the merge cell first. Missing: {hs_merged_path}")
ds_hs = xr.open_dataset(hs_merged_path)
hs_var = "hs" if "hs" in ds_hs.data_vars else list(ds_hs.data_vars)[0]
hs_da = ds_hs[hs_var]

# Fixed colorbar across all timesteps: use 99th percentile of entire dataset
all_vals = hs_da.values
HS_VMAX = float(np.nanpercentile(all_vals, 99)) if np.size(all_vals) > 0 else 0.01
HS_VMAX = max(HS_VMAX, 0.01)  # avoid vmax=0 (collapsed colorbar)

def plot_hs_map(cyclone_idx=0, hour_idx=0):
    """Plot Hs on map for selected cyclone and hour (merged grids)."""
    fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={"projection": ccrs.PlateCarree()})
    extent = [-81.0, -74.0, 32.0, 37.5]
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor="0.9", linewidth=0)
    ax.add_feature(cfeature.OCEAN, facecolor="lightsteelblue", alpha=0.5, linewidth=0)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8, edgecolor="k")

    # If cyclone_id dimension exists, use it; otherwise fall back to original behaviour
    if "cyclone_id" in hs_da.dims:
        hs_slice = hs_da.isel(cyclone_id=cyclone_idx, time=hour_idx)
    else:
        hs_slice = hs_da.isel(time=hour_idx)

    lon = hs_da.lon.values if hasattr(hs_da, 'lon') else ds_hs.lon.values
    lat = hs_da.lat.values if hasattr(hs_da, 'lat') else ds_hs.lat.values
    vals = hs_slice.values
    vmax_data = np.nanmax(vals) if vals.size > 0 else 0
    if vmax_data < 1e-6:
        ax.set_title(
            f"Significant wave height (Hs) - cyclone {cyclone_idx}, hour {hour_idx} "
            "[WARNING: all zeros - check inputs]"
        )

    # Fixed colorbar for all timesteps (99th percentile of full dataset)
    sc = ax.scatter(
        lon,
        lat,
        c=vals,
        s=12,
        cmap="viridis",
        transform=ccrs.PlateCarree(),
        vmin=0,
        vmax=HS_VMAX,
    )

    cbar = plt.colorbar(sc, ax=ax, shrink=0.6)
    cbar.set_label("Hs (m)")
    if "cyclone_id" in hs_da.dims:
        ax.set_title(f"Significant wave height (Hs) - cyclone {cyclone_idx}, hour {hour_idx} (merged)")
    else:
        ax.set_title(f"Significant wave height (Hs) - sea state {hour_idx} (merged)")
    ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5, linestyle="--")
    plt.tight_layout()
    plt.show()

# Slider limits
if "cyclone_id" in hs_da.dims:
    n_cyc = hs_da.sizes["cyclone_id"]
else:
    n_cyc = 1
n_hours = hs_da.sizes["time"]

if USE_WIDGETS:
    interact(
        plot_hs_map,
        cyclone_idx=IntSlider(min=0, max=n_cyc - 1, value=0, description="cyclone_id"),
        hour_idx=IntSlider(min=0, max=n_hours - 1, value=0, description="hour"),
    )
else:
    plot_hs_map(cyclone_idx=0, hour_idx=0)

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# --- Config: choose cyclone to plot ---
cyclone_idx = 4  # change this integer to select cyclone_id

# --- Load merged fields ---
_base = BASE_DIR if isinstance(BASE_DIR, str) else str(BASE_DIR)
MERGED_DIR = os.path.join(_base, "outputs", "merged_grids")

ds_hs = xr.open_dataset(os.path.join(MERGED_DIR, "hs_merged_all.nc"))
ds_tp = xr.open_dataset(os.path.join(MERGED_DIR, "tp_merged_all.nc"))
ds_dp = xr.open_dataset(os.path.join(MERGED_DIR, "dp_merged_all.nc"))

hs_da = ds_hs["hs"] if "hs" in ds_hs.data_vars else list(ds_hs.data_vars.values())[0]
tp_da = ds_tp["tp"] if "tp" in ds_tp.data_vars else list(ds_tp.data_vars.values())[0]
dp_da = ds_dp["dp"] if "dp" in ds_dp.data_vars else list(ds_dp.data_vars.values())[0]

# Handle cyclone_id dimension (if present)
def select_cyclone(da, cid):
    if "cyclone_id" in da.dims:
        return da.isel(cyclone_id=cid)
    return da  # fallback: no cyclone dimension

hs_cyc = select_cyclone(hs_da, cyclone_idx)
tp_cyc = select_cyclone(tp_da, cyclone_idx)
dp_cyc = select_cyclone(dp_da, cyclone_idx)

lon = hs_cyc.lon.values if hasattr(hs_cyc, "lon") else ds_hs.lon.values
lat = hs_cyc.lat.values if hasattr(hs_cyc, "lat") else ds_hs.lat.values

n_hours = hs_cyc.sizes["time"]

# Global color scales for consistency
HS_VMAX = float(np.nanpercentile(hs_cyc.values, 99))
TP_VMAX = float(np.nanpercentile(tp_cyc.values, 99))
HS_VMAX = max(HS_VMAX, 0.01)
TP_VMAX = max(TP_VMAX, 0.1)

extent = [-81.0, -74.0, 32.0, 37.5]

def _make_figure(data_cyc, vmin, vmax, cmap, title_prefix):
    fig, axes = plt.subplots(
        5, 5, figsize=(15, 15),
        subplot_kw={"projection": ccrs.PlateCarree()},
    )
    axes = axes.ravel()

    for h in range(25):  # 0..24, last panel may be empty
        ax = axes[h]
        ax.set_extent(extent, crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.LAND, facecolor="0.9", linewidth=0)
        ax.add_feature(cfeature.OCEAN, facecolor="lightsteelblue", alpha=0.5, linewidth=0)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor="k")

        if h < n_hours:
            vals = data_cyc.isel(time=h).values
            sc = ax.scatter(
                lon, lat, c=vals, s=8,
                cmap=cmap, vmin=vmin, vmax=vmax,
                transform=ccrs.PlateCarree(),
            )
            ax.set_title(f"{title_prefix} – h={h}")
        else:
            ax.set_axis_off()

    # Single colorbar for the whole figure
    cax = fig.add_axes([0.25, 0.05, 0.5, 0.02])
    norm = plt.cm.colors.Normalize(vmin=vmin, vmax=vmax)
    cb = plt.colorbar(
        plt.cm.ScalarMappable(norm=norm, cmap=cmap),
        cax=cax, orientation="horizontal",
    )
    cb.set_label(title_prefix)
    fig.suptitle(f"{title_prefix} – cyclone_id={cyclone_idx}", y=0.98, fontsize=14)
    plt.tight_layout(rect=[0, 0.08, 1, 0.96])
    plt.show()

# 1) Hs maps (5x5 for 24 hours)
_make_figure(hs_cyc, vmin=0.0, vmax=HS_VMAX, cmap="viridis", title_prefix="Hs (m)")

# 2) Tp maps
_make_figure(tp_cyc, vmin=0.0, vmax=TP_VMAX, cmap="plasma", title_prefix="Tp (s)")

# 3) Direction maps (0–360)
_make_figure(dp_cyc, vmin=0.0, vmax=360.0, cmap="twilight", title_prefix="Dp (deg)")